# Example: SISO Frequency Response Estimation

This example demonstrates how to estimate the frequency response of a
simple SISO system using `sid.freq_bt` (Blackman-Tukey method) and plot
the results with confidence bands.

Translated from `exampleSISO.m`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
import sid

%matplotlib inline

## Generate test data

True system: $G(z) = 1 / (1 - 0.9\,z^{-1})$

This is a stable first-order system with a pole at $z = 0.9$.

In [ ]:
rng = np.random.default_rng(42)

N = 1000                                    # Number of samples
Ts = 0.01                                   # Sample time (seconds)
u = rng.standard_normal(N)                  # White noise input
y_clean = lfilter([1], [1, -0.9], u)        # Noiseless output
noise = 0.1 * rng.standard_normal(N)        # Measurement noise
y = y_clean + noise                         # Noisy output

## Estimate frequency response using Blackman-Tukey

In [ ]:
result = sid.freq_bt(y, u, sample_time=Ts)

## Plot Bode diagram

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6))
sid.bode_plot(result)
plt.tight_layout()
plt.show()

## Plot noise spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sid.spectrum_plot(result)
plt.tight_layout()
plt.show()

## Compare different window sizes

Larger window = finer resolution but more variance.

In [ ]:
r10  = sid.freq_bt(y, u, window_size=10,  sample_time=Ts)
r30  = sid.freq_bt(y, u, window_size=30,  sample_time=Ts)
r100 = sid.freq_bt(y, u, window_size=100, sample_time=Ts)

freq = r30.frequency / Ts

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(freq, 20 * np.log10(np.abs(r10.response)),  'b', label='M = 10')
ax.semilogx(freq, 20 * np.log10(np.abs(r30.response)),  'r', label='M = 30')
ax.semilogx(freq, 20 * np.log10(np.abs(r100.response)), 'g', label='M = 100')
ax.set_xlabel('Frequency (rad/s)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('Effect of Window Size on Frequency Resolution')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Preprocessing: detrend data before estimation

Add a drift to the data and show that detrending improves results.

In [ ]:
y_drift = y + 0.01 * np.arange(1, N + 1)   # add linear drift
u_drift = u + 5                              # add DC offset to input

# Without detrending: drift biases the low-frequency estimate
result_raw = sid.freq_bt(y_drift, u_drift, sample_time=Ts)

# With detrending
y_dt, _ = sid.detrend(y_drift)
u_dt, _ = sid.detrend(u_drift)
result_dt = sid.freq_bt(y_dt, u_dt, sample_time=Ts)

print(f'Without detrend: max |G| at low freq = {np.max(np.abs(result_raw.response)):.2f}')
print(f'With detrend:    max |G| at low freq = {np.max(np.abs(result_dt.response)):.2f}')

## Model validation: residual analysis

In [ ]:
resid = sid.residual(result, y, u)

print('Whiteness test:',    'PASS' if resid.whiteness_pass    else 'FAIL')
print('Independence test:', 'PASS' if resid.independence_pass else 'FAIL')

## Model validation: compare predicted vs measured

In [ ]:
comp = sid.compare(result, y, u)
print(f'NRMSE fit: {comp.fit[0]:.1f}%')

## Time series mode (no input)

Estimate the output power spectrum of an AR(1) process.

In [ ]:
y_ts = lfilter([1], [1, -0.8], rng.standard_normal(500))
result_ts = sid.freq_bt(y_ts, None)

sid.spectrum_plot(result_ts)
plt.title('AR(1) Output Spectrum')
plt.tight_layout()
plt.show()